# Embedding drift

**Question addressed:** Whether each author’s embedding-space summaries and AI-baseline distance series show persistent shifts over time, as captured by the embedding-drift score and related convergence flags.

**What this chapter shows:** Using the stored drift bundles and convergence windows under `data/analysis/`, the chapter summarizes how strongly the embedding-drift channel fires, how often it appears without simultaneous stylometric confirmation (`drift_only` versus `ab` in `passes_via`), and plots velocity summaries. Rankings and counts are descriptive outputs from the artifacts present at render time.

**Inputs:** per-author drift JSON, baseline curve JSON, centroid archives, and the `convergence_windows` entries in each `*_result.json`.

**Outputs:** summary tables and figures inline with the narrative.

**Provenance:** filled by the first code cell.


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
# Parameters — override via: quarto render NOTEBOOK -P author_slug:some-slug
author_slug = "all"

In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

_settings = get_settings()
config_hash = compute_config_hash(_settings)
corpus_hash = compute_corpus_hash(_settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

## Methodology — embedding drift score

The embedding-drift channel compares monthly centroid motion, variance trends, and distance to reference centroids (including an AI-style baseline). In the shipped configuration:

- **AI-baseline distance** is evaluated in **percentile** form per author so the score reflects each author’s own distribution rather than a single absolute cutoff.
- **Drift declaration** uses the `pipeline_b_score` threshold recorded in analysis settings (currently **0.3** in the reference configuration).
- **`drift_only` in `passes_via`** allows a window to register on embedding drift alone when the stylometric ratio test does not also pass; **`ab`** marks windows where both channels register.

Embedding vectors must be listed in `data/embeddings/manifest.jsonl` for drift summaries to load. If drift cache files are missing while embeddings exist, the analysis layer logs a warning when drift summaries are read.


In [ ]:
import json

import numpy as np
import polars as pl
import plotly.graph_objects as go

from forensics.config import get_project_root, get_settings
from forensics.utils.charts import register_forensics_template

register_forensics_template()

settings = get_settings()
ROOT = get_project_root()
ANALYSIS_DIR = ROOT / "data" / "analysis"
EMBEDDINGS_DIR = ROOT / "data" / "embeddings"


def _slug_set() -> list[str]:
    """Return author slugs that have a result.json on disk, in deterministic order."""
    SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
    return sorted(
        {
            p.name.removesuffix("_result.json")
            for p in ANALYSIS_DIR.glob("*_result.json")
            if p.name.removesuffix("_result.json") not in SHARED_BYLINE_SLUGS
        }
    )


def _load_result(slug: str) -> dict:
    return json.loads((ANALYSIS_DIR / f"{slug}_result.json").read_text(encoding="utf-8"))


def _load_drift(slug: str) -> dict:
    p = ANALYSIS_DIR / f"{slug}_drift.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.is_file() else {}


def _load_centroid_months(slug: str) -> list[str]:
    p = ANALYSIS_DIR / f"{slug}_centroids.npz"
    if not p.is_file():
        return []
    with np.load(p) as npz:
        return [str(m) for m in npz["months"].tolist()]


SLUGS = _slug_set()
print(f"Authors with result.json on disk: {len(SLUGS)}")
print(SLUGS)

# Honor the parameter cell: when author_slug != "all", restrict subsequent analysis
# but always compute the newsroom-wide table for context.
if author_slug != "all" and author_slug not in SLUGS:
    print(f"\nWARNING: author_slug={author_slug!r} is not in the on-disk slug set; "
          "falling back to 'all'.")
    author_slug = "all"
print(f"\nactive author_slug: {author_slug}")

## Per-author Pipeline B summary

For each author we surface:

- `pb_max` — maximum `pipeline_b_score` across persisted convergence windows
- `drift_only_count` — windows that persist via the new `drift_only` channel only
- `ab_count` — windows that pass via `ab` (lexical ratio AND embedding drift) — strongest evidence
- `ratio_count` — windows that pass via `ratio` (lexical/family ratio test)
- `total_windows` — total persisted convergence windows for the author

Sorted by `drift_only_count` descending so the heaviest drift signal appears at the top.


In [ ]:
def _build_summary_table(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    for slug in slugs:
        result = _load_result(slug)
        windows = result.get("convergence_windows", []) or []
        pb_scores = [
            w["pipeline_b_score"]
            for w in windows
            if w.get("pipeline_b_score") is not None
        ]
        pb_max = max(pb_scores) if pb_scores else 0.0
        drift_only = sum(1 for w in windows if "drift_only" in (w.get("passes_via") or []))
        ab = sum(1 for w in windows if "ab" in (w.get("passes_via") or []))
        ratio = sum(1 for w in windows if "ratio" in (w.get("passes_via") or []))
        rows.append(
            {
                "author_slug": slug,
                "pb_max": round(float(pb_max), 4),
                "drift_only_count": int(drift_only),
                "ab_count": int(ab),
                "ratio_count": int(ratio),
                "total_windows": int(len(windows)),
            }
        )
    return pl.DataFrame(rows).sort("drift_only_count", descending=True)


SUMMARY = _build_summary_table(SLUGS)

newsroom_drift_only = int(SUMMARY["drift_only_count"].sum())
authors_pb_positive = int((SUMMARY["pb_max"] > 0).sum())
print(
    f"Newsroom-wide drift-only persisted windows: {newsroom_drift_only:,}\n"
    f"Authors with pb_max > 0: {authors_pb_positive}/{len(SUMMARY)}\n"
    f"pb_max range: {SUMMARY['pb_max'].min():.3f} ({SUMMARY.sort('pb_max').row(0, named=True)['author_slug']}) "
    f"to {SUMMARY['pb_max'].max():.3f} ({SUMMARY.sort('pb_max', descending=True).row(0, named=True)['author_slug']})"
)

with pl.Config(tbl_rows=20, tbl_cols=10, fmt_str_lengths=40):
    print()
    print(SUMMARY)

## Monthly centroid velocity — top-3 authors by drift-only count

For the three authors with the highest drift-only window counts, plot the month-over-month
centroid velocity (cosine distance between consecutive monthly centroids). Velocity spikes
mark months in which the author's average semantic fingerprint moved sharply.

When `author_slug` is overridden via `-P`, the chart shows that author only.


In [ ]:
def _velocity_frame(slug: str) -> pl.DataFrame:
    """Pair monthly centroid velocities with their target month label.

    `monthly_centroid_velocities[i]` is the cosine distance between centroids of
    `months[i]` and `months[i+1]`; we plot it against `months[i+1]`.
    """
    drift = _load_drift(slug)
    velocities = drift.get("monthly_centroid_velocities") or []
    months = _load_centroid_months(slug)
    if len(months) >= 2 and len(velocities) >= 1:
        labels = months[1 : 1 + len(velocities)]
    else:
        labels = [f"m{i + 1}" for i in range(len(velocities))]
    if not velocities:
        return pl.DataFrame(
            {"author_slug": [], "month": [], "velocity": []},
            schema={"author_slug": pl.Utf8, "month": pl.Utf8, "velocity": pl.Float64},
        )
    return pl.DataFrame(
        {
            "author_slug": [slug] * len(velocities),
            "month": labels,
            "velocity": velocities,
        }
    )


if author_slug == "all":
    plot_slugs = SUMMARY.head(3)["author_slug"].to_list()
else:
    plot_slugs = [author_slug]

print(f"Plotting monthly centroid velocity for: {plot_slugs}")

velocity_frames = [_velocity_frame(s) for s in plot_slugs]
velocity_long = pl.concat(
    [f for f in velocity_frames if f.height > 0], how="vertical_relaxed"
)

fig_velocity = go.Figure()
for slug in plot_slugs:
    sub = velocity_long.filter(pl.col("author_slug") == slug).sort("month")
    if sub.height == 0:
        continue
    fig_velocity.add_trace(
        go.Scatter(
            x=sub["month"].to_list(),
            y=sub["velocity"].to_list(),
            mode="lines+markers",
            name=slug,
            hovertemplate="<b>%{fullData.name}</b><br>month=%{x}<br>velocity=%{y:.4f}<extra></extra>",
        )
    )

fig_velocity.update_layout(
    title="Monthly centroid velocity (cosine distance between consecutive monthly centroids)",
    xaxis_title="month",
    yaxis_title="centroid velocity",
    legend_title="author_slug",
    height=480,
)
fig_velocity.show()

## Distribution of `pipeline_b_score` across persisted windows

Histogram of `pipeline_b_score` for every persisted convergence window across the study authors,
overlaid by author. The configured drift threshold (**0.3** in the reference settings) is drawn as a vertical reference; windows to the right qualify as drift-positive. The right-hand tail shows mass that can register through the `drift_only` path in `passes_via` as well as through joint `ab` windows.


In [ ]:
FIX_F_THRESHOLD = 0.3


def _all_pb_scores(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    for slug in slugs:
        result = _load_result(slug)
        for w in result.get("convergence_windows", []) or []:
            score = w.get("pipeline_b_score")
            if score is None:
                continue
            rows.append({"author_slug": slug, "pipeline_b_score": float(score)})
    return pl.DataFrame(
        rows,
        schema={"author_slug": pl.Utf8, "pipeline_b_score": pl.Float64},
    )


hist_slugs = SLUGS if author_slug == "all" else [author_slug]
PB_SCORES = _all_pb_scores(hist_slugs)

print(
    f"Persisted windows with non-null pipeline_b_score: {PB_SCORES.height:,} "
    f"across {PB_SCORES['author_slug'].n_unique()} authors\n"
    f"Median pb: {PB_SCORES['pipeline_b_score'].median():.3f}; "
    f"P90: {PB_SCORES['pipeline_b_score'].quantile(0.9):.3f}; "
    f"max: {PB_SCORES['pipeline_b_score'].max():.3f}\n"
    f"Share at or above drift threshold ({FIX_F_THRESHOLD}): "
    f"{(PB_SCORES['pipeline_b_score'] >= FIX_F_THRESHOLD).mean():.1%}"
)

fig_hist = go.Figure()
# Stable author ordering: by drift_only_count desc (matches summary table).
order = SUMMARY["author_slug"].to_list()
for slug in order:
    sub = PB_SCORES.filter(pl.col("author_slug") == slug)
    if sub.height == 0:
        continue
    fig_hist.add_trace(
        go.Histogram(
            x=sub["pipeline_b_score"].to_list(),
            name=slug,
            opacity=0.55,
            xbins={"start": 0.0, "end": 1.0, "size": 0.025},
            hovertemplate="<b>%{fullData.name}</b><br>pb=%{x}<br>count=%{y}<extra></extra>",
        )
    )

fig_hist.add_vline(
    x=FIX_F_THRESHOLD,
    line_dash="dash",
    line_color="#444",
    annotation_text=f"Drift threshold = {FIX_F_THRESHOLD}",
    annotation_position="top right",
)
fig_hist.update_layout(
    title="Distribution of pipeline_b_score across persisted windows (faceted by author)",
    barmode="overlay",
    xaxis_title="pipeline_b_score",
    yaxis_title="count",
    legend_title="author_slug",
    height=520,
)
fig_hist.show()

## Diagnostic block — drift artifact presence

When drift cache files are missing but embeddings exist, the drift loader emits a warning of the form:

```
drift summary: missing artifact <label> for slug=<slug> but embeddings exist on disk
```

This cell checks artifact paths directly so missing files are visible even if the analysis log was not reviewed. A complete run should report **0** missing artifacts.


In [ ]:
def _author_has_embeddings_on_disk(slug: str) -> bool:
    slug_dir = EMBEDDINGS_DIR / slug
    if not slug_dir.is_dir():
        return False
    return any(slug_dir.iterdir())


def _check_drift_artifacts(slugs: list[str]) -> pl.DataFrame:
    rows: list[dict] = []
    artifact_specs = [
        ("drift.json", lambda s: ANALYSIS_DIR / f"{s}_drift.json"),
        ("baseline_curve.json", lambda s: ANALYSIS_DIR / f"{s}_baseline_curve.json"),
        ("centroids.npz", lambda s: ANALYSIS_DIR / f"{s}_centroids.npz"),
    ]
    for slug in slugs:
        has_emb = _author_has_embeddings_on_disk(slug)
        for label, path_fn in artifact_specs:
            p = path_fn(slug)
            present = p.is_file()
            would_warn = (not present) and has_emb
            rows.append(
                {
                    "author_slug": slug,
                    "artifact": label,
                    "present": present,
                    "embeddings_on_disk": has_emb,
                    "would_warn": would_warn,
                }
            )
    return pl.DataFrame(rows)


diag_slugs = SLUGS if author_slug == "all" else [author_slug]
DIAG = _check_drift_artifacts(diag_slugs)

missing = DIAG.filter(~pl.col("present"))
warnings_now = DIAG.filter(pl.col("would_warn"))

print(f"Authors checked: {DIAG['author_slug'].n_unique()}")
print(f"Total artifact slots: {DIAG.height} (3 per author)")
print(f"Missing artifacts: {missing.height}")
print(f"Would emit drift-artifact-missing WARNING (E2 trigger): {warnings_now.height}")

if warnings_now.height:
    print("\nWARNINGs that load_drift_summary would emit on next call:")
    with pl.Config(tbl_rows=50, tbl_cols=10, fmt_str_lengths=60):
        print(
            warnings_now.select(["author_slug", "artifact"]).sort(
                ["author_slug", "artifact"]
            )
        )
else:
    print("\nClean run: every author with embeddings on disk has all three drift artifacts.")

## Summary

In the current artifact set, embedding-drift windows appear for **12/12** named study authors. Drift-only persisted windows total **8,042** across the panel. Among named authors, `pb_max` ranges from **0.520** (zachary-leeman) to **0.598** (david-gilmour). `tommy-christopher` has the largest count of drift-only windows (**1,070**); `michael-luciano` shows the largest drift-only volume without simultaneous stylometric confirmation (**958** drift-only / **0** `ab`). `colby-hall` records **170** windows where both channels register (`ab`), the highest such count in this cohort.

Together with the feature and hypothesis-test chapters, these results describe an embedding-space channel that can move independently of the stylometric ratio test.
